# GOES Aggregates Used by the Candidate Model

This notebook visualizes the actual GOES daily-aggregate features used by the event-candidate model for one `fire_id/date` prediction case.

It shows:

- VIIRS current burned/fire mask at day `d` and next-day growth target `d+1`.
- Full 256x256 GOES daily aggregate maps for `lag0=d`, `lag1=d-1`, ... .
- Candidate-level sparse maps showing the values actually attached to candidate pixels.
- Positive vs negative candidate feature means for the selected case.

Use this to inspect whether the model is seeing useful aggregate signals for the next-day prediction.

In [ ]:
# Configuration
from pathlib import Path

REPO_ROOT = Path('/home/jlc3q/New_project/TS-Agentic-AI')
CANDIDATE_ROOT = Path('/home/jlc3q/data/SatFire/event_candidates')
GOES_ROOT = Path('/home/jlc3q/data/GOES_clipped_tif_common_wgs84')

SPLIT = 'test'             # train, val, or test
HISTORY_DAYS = 4           # 2 or 4; use 6 after h6 files exist
CONNECTIVITY = 8
CANDIDATE_RADIUS = 5.0
MIN_COMPONENT_PIXELS = 1
HIGH_FRP_PERCENTILE = 90.0

# Leave as None to automatically choose the fire/date with the most positive candidate pixels.
FIRE_ID = None
DATE = None

# Plot controls
CASE_RANK = 0              # 0 = most positives, 1 = second most positives, ... when FIRE_ID/DATE are None
MAX_SCATTER_POINTS = 25000
RANDOM_SEED = 42

In [ ]:
import sys
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# Make legacy scripts importable in notebook execution.
for p in [REPO_ROOT / 'legacy', REPO_ROOT / 'legacy' / 'scripts']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from analyze_pred_event_windows import load_daily_masks, resolve_locations
from join_pred_event_candidates_goes_frp import GoesFrpCache, date_lag

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 160

In [ ]:
def radius_tag(value: float) -> str:
    return str(value).replace('.', 'p')


def history_tag(history_days: int) -> str:
    return f'_h{history_days}' if history_days > 1 else ''


def candidate_csv(split: str, history_days: int) -> Path:
    suffix = (
        f'{split}_conn{CONNECTIVITY}_r{radius_tag(CANDIDATE_RADIUS)}'
        f'_mincomp{MIN_COMPONENT_PIXELS}{history_tag(history_days)}'
    )
    return CANDIDATE_ROOT / f'pred_event_candidates_{suffix}_goes_frp.csv'


def summarize_cases(path: Path, chunksize: int = 300_000) -> pd.DataFrame:
    cols = ['fire_id', 'date', 'day_idx', 'label_ignited_next_day']
    parts = []
    for chunk in pd.read_csv(path, usecols=cols, chunksize=chunksize):
        g = chunk.groupby(['fire_id', 'date', 'day_idx'], sort=False)['label_ignited_next_day'].agg(['size', 'sum'])
        parts.append(g.reset_index().rename(columns={'size': 'candidate_pixels', 'sum': 'positive_pixels'}))
    out = pd.concat(parts, ignore_index=True)
    out = out.groupby(['fire_id', 'date', 'day_idx'], as_index=False).sum()
    out['positive_rate'] = out['positive_pixels'] / out['candidate_pixels']
    return out.sort_values(['positive_pixels', 'positive_rate', 'candidate_pixels'], ascending=[False, False, False])


def load_candidate_slice(path: Path, fire_id: str, date: str, chunksize: int = 300_000) -> pd.DataFrame:
    parts = []
    for chunk in pd.read_csv(path, chunksize=chunksize):
        hit = chunk[(chunk['fire_id'].astype(str) == str(fire_id)) & (chunk['date'].astype(str) == str(date))]
        if len(hit):
            parts.append(hit.copy())
    if not parts:
        raise ValueError(f'No candidate rows found for fire_id={fire_id}, date={date} in {path}')
    return pd.concat(parts, ignore_index=True)


def find_label_sel(split: str, fire_id: str):
    pairs = list(resolve_locations(split))
    for fid, label_sel in pairs:
        if str(fid) == str(fire_id):
            return label_sel
    # Some downstream files may include/remove the US_ prefix; handle both conservatively.
    alt = str(fire_id).replace('US_', '')
    for fid, label_sel in pairs:
        if str(fid).replace('US_', '') == alt:
            return label_sel
    raise ValueError(f'Could not find label selector for {fire_id} in split={split}')


def load_viirs_masks_for_case(split: str, fire_id: str):
    label_sel = find_label_sel(split, fire_id)
    return load_daily_masks(fire_id, label_sel)


def robust_vmax(arr, q=99):
    values = np.asarray(arr, dtype=np.float32)
    values = values[np.isfinite(values)]
    values = values[values > 0]
    if values.size == 0:
        return 1.0
    return float(np.nanpercentile(values, q)) or 1.0


def sparse_candidate_map(df: pd.DataFrame, field: str, fill=np.nan) -> np.ndarray:
    arr = np.full((256, 256), fill, dtype=np.float32)
    rr = df['candidate_row'].to_numpy(dtype=np.int64)
    cc = df['candidate_col'].to_numpy(dtype=np.int64)
    vals = df[field].fillna(0).to_numpy(dtype=np.float32)
    # Candidate rows should be unique per date after nearest-component assignment, but max is safe.
    if np.isnan(fill):
        arr[rr, cc] = vals
    else:
        np.maximum.at(arr, (rr, cc), vals)
    return arr


def plot_grid(maps, titles, cmap='magma', vmax=None, ncols=4, figsize=None, overlay_mask=None):
    n = len(maps)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    if figsize is None:
        figsize = (4.0 * ncols, 3.8 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')
    for ax, arr, title in zip(axes.ravel(), maps, titles):
        arr = np.asarray(arr, dtype=np.float32)
        local_vmax = robust_vmax(arr) if vmax is None else vmax
        im = ax.imshow(arr, cmap=cmap, vmin=0, vmax=local_vmax)
        if overlay_mask is not None:
            ax.contour(overlay_mask.astype(float), levels=[0.5], colors='cyan', linewidths=0.5)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    plt.tight_layout()
    return fig

In [ ]:
path = candidate_csv(SPLIT, HISTORY_DAYS)
print('candidate file:', path)
if not path.exists():
    raise FileNotFoundError(path)

if FIRE_ID is None or DATE is None:
    cases = summarize_cases(path)
    display(cases.head(12))
    selected = cases.iloc[CASE_RANK]
    FIRE_ID = str(selected['fire_id'])
    DATE = str(selected['date'])
    print(f'Auto-selected case rank={CASE_RANK}: fire_id={FIRE_ID}, date={DATE}, positives={int(selected.positive_pixels)}, candidates={int(selected.candidate_pixels)}')
else:
    print(f'Using configured case: fire_id={FIRE_ID}, date={DATE}')

df_case = load_candidate_slice(path, FIRE_ID, DATE)
print('candidate rows:', len(df_case))
print('positive candidates:', int(df_case['label_ignited_next_day'].sum()))
print('positive rate:', float(df_case['label_ignited_next_day'].mean()))
print('day_idx:', sorted(df_case['day_idx'].unique()))

dates, masks = load_viirs_masks_for_case(SPLIT, FIRE_ID)
day_idx = int(df_case['day_idx'].iloc[0])
current_mask = masks[day_idx].astype(bool)
future_mask = masks[day_idx + 1].astype(bool)
growth_mask = future_mask & ~current_mask
print('VIIRS date check:', dates[day_idx], '->', dates[day_idx + 1])
print('current fire pixels:', int(current_mask.sum()), 'growth pixels:', int(growth_mask.sum()))

In [ ]:
# VIIRS current fire, next-day growth, and candidate positions.
rng = np.random.default_rng(RANDOM_SEED)
plot_df = df_case
if len(plot_df) > MAX_SCATTER_POINTS:
    plot_df = plot_df.sample(MAX_SCATTER_POINTS, random_state=RANDOM_SEED)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].imshow(current_mask, cmap='gray')
axes[0].set_title(f'VIIRS current fire mask\n{FIRE_ID} {DATE}')
axes[1].imshow(growth_mask, cmap='hot')
axes[1].set_title(f'VIIRS next-day growth target\n{dates[day_idx + 1]}')

base = np.zeros((256, 256, 3), dtype=np.float32)
base[current_mask] = [0.35, 0.35, 0.35]
base[growth_mask] = [1.0, 0.15, 0.05]
axes[2].imshow(base)
neg = plot_df[plot_df['label_ignited_next_day'] == 0]
pos = plot_df[plot_df['label_ignited_next_day'] == 1]
axes[2].scatter(neg['candidate_col'], neg['candidate_row'], s=1, c='deepskyblue', alpha=0.12, label='candidate negative')
axes[2].scatter(pos['candidate_col'], pos['candidate_row'], s=3, c='yellow', alpha=0.85, label='candidate positive')
axes[2].set_title('Candidate pixels used by model\nblue=negative, yellow=positive')
axes[2].legend(loc='lower right', fontsize=8)
for ax in axes:
    ax.axis('off')
plt.tight_layout()

In [ ]:
# Full 256x256 GOES aggregate maps for lag0=d, lag1=d-1, ... .
cache = GoesFrpCache(GOES_ROOT)
lag_features = []
for lag in range(HISTORY_DAYS):
    lag_date = date_lag(DATE, lag)
    features = cache.get_day_features(FIRE_ID, lag_date, HIGH_FRP_PERCENTILE)
    lag_features.append((lag, lag_date, features))
    print(f'lag{lag}: {lag_date}, frp_total_log1p={features["goes_frp_total_log1p"]:.3f}, weighted_active_total={features["goes_weighted_active_total"]:.3f}')

In [ ]:
# GOES FRP sum maps by lag. Cyan contour = current VIIRS fire perimeter.
frp_maps = [item[2]['frp_sum'] for item in lag_features]
frp_titles = [f'lag{lag} {lag_date}\nGOES frp_sum_log1p' for lag, lag_date, _ in lag_features]
plot_grid(frp_maps, frp_titles, cmap='inferno', ncols=HISTORY_DAYS, overlay_mask=current_mask)

In [ ]:
# GOES weighted-active maps by lag. This is active mask suppressed/amplified by FRP strength.
wa_maps = [item[2]['weighted_active'] for item in lag_features]
wa_titles = [f'lag{lag} {lag_date}\nGOES weighted_active' for lag, lag_date, _ in lag_features]
plot_grid(wa_maps, wa_titles, cmap='viridis', ncols=HISTORY_DAYS, overlay_mask=current_mask)

In [ ]:
# GOES history summary maps computed from the lag maps.
frp_stack = np.stack(frp_maps, axis=0)
wa_stack = np.stack(wa_maps, axis=0)
summary_maps = [
    frp_stack.sum(axis=0),
    frp_stack.max(axis=0),
    frp_stack.mean(axis=0),
    wa_stack.sum(axis=0),
    wa_stack.max(axis=0),
    wa_stack.mean(axis=0),
]
summary_titles = [
    f'h{HISTORY_DAYS} frp_sum hist_sum',
    f'h{HISTORY_DAYS} frp_sum hist_max',
    f'h{HISTORY_DAYS} frp_sum hist_mean',
    f'h{HISTORY_DAYS} weighted_active hist_sum',
    f'h{HISTORY_DAYS} weighted_active hist_max',
    f'h{HISTORY_DAYS} weighted_active hist_mean',
]
plot_grid(summary_maps, summary_titles, cmap='magma', ncols=3, overlay_mask=current_mask)

In [ ]:
# Candidate-level sparse maps: these are the exact feature values attached to candidate rows.
# For HISTORY_DAYS > 1, lag columns are present. If not, fallback to current-day base columns.
frp_candidate_fields = []
weighted_candidate_fields = []
for lag in range(HISTORY_DAYS):
    frp_col = f'goes_frp_sum_at_candidate_lag{lag}'
    wa_col = f'goes_weighted_active_at_candidate_lag{lag}'
    if frp_col in df_case.columns:
        frp_candidate_fields.append(frp_col)
    if wa_col in df_case.columns:
        weighted_candidate_fields.append(wa_col)
if not frp_candidate_fields:
    frp_candidate_fields = ['goes_frp_sum_at_candidate']
if not weighted_candidate_fields:
    weighted_candidate_fields = ['goes_weighted_active_at_candidate']

maps = [sparse_candidate_map(df_case, col) for col in frp_candidate_fields]
titles = [col.replace('goes_', '').replace('_at_candidate_', '\n') for col in frp_candidate_fields]
plot_grid(maps, titles, cmap='inferno', ncols=min(HISTORY_DAYS, 4), overlay_mask=current_mask)

maps = [sparse_candidate_map(df_case, col) for col in weighted_candidate_fields]
titles = [col.replace('goes_', '').replace('_at_candidate_', '\n') for col in weighted_candidate_fields]
plot_grid(maps, titles, cmap='viridis', ncols=min(HISTORY_DAYS, 4), overlay_mask=current_mask)

In [ ]:
# Candidate-level local 5x5 maps used by the model.
local_fields = [
    'goes_frp_sum_5x5_max',
    'goes_frp_sum_5x5_mean',
    'goes_weighted_active_5x5_max',
    'goes_weighted_active_5x5_mean',
    'goes_distance_to_high_frp',
]
local_fields = [c for c in local_fields if c in df_case.columns]
plot_grid(
    [sparse_candidate_map(df_case, c) for c in local_fields],
    [c.replace('goes_', '').replace('_', ' ') for c in local_fields],
    cmap='plasma',
    ncols=3,
    overlay_mask=current_mask,
)

In [ ]:
# Positive vs negative candidate feature means for this one fire/date.
inspect_cols = [
    'distance_px',
    'history_candidate_active_days',
    'history_nearest_active_days',
    'goes_frp_sum_at_candidate',
    'goes_frp_max_at_candidate',
    'goes_weighted_active_at_candidate',
    'goes_frp_sum_5x5_max',
    'goes_weighted_active_5x5_max',
]
for lag in range(HISTORY_DAYS):
    inspect_cols.extend([
        f'goes_frp_sum_at_candidate_lag{lag}',
        f'goes_weighted_active_at_candidate_lag{lag}',
    ])
inspect_cols = [c for c in dict.fromkeys(inspect_cols) if c in df_case.columns]

summary = df_case.groupby('label_ignited_next_day')[inspect_cols].mean().T
summary.columns = ['negative_mean' if c == 0 else 'positive_mean' for c in summary.columns]
if 'negative_mean' in summary.columns and 'positive_mean' in summary.columns:
    summary['positive_minus_negative'] = summary['positive_mean'] - summary['negative_mean']
    summary['positive_over_negative'] = summary['positive_mean'] / summary['negative_mean'].replace(0, np.nan)
display(summary.sort_values('positive_minus_negative', ascending=False) if 'positive_minus_negative' in summary.columns else summary)

In [ ]:
# Optional: compare h2 vs h4 candidate files for the same fire/date if both exist.
compare_rows = []
for h in [2, 4, 6]:
    p = candidate_csv(SPLIT, h)
    if not p.exists():
        continue
    try:
        dfx = load_candidate_slice(p, FIRE_ID, DATE)
    except ValueError:
        continue
    compare_rows.append({
        'history_days': h,
        'rows': len(dfx),
        'positives': int(dfx['label_ignited_next_day'].sum()),
        'positive_rate': float(dfx['label_ignited_next_day'].mean()),
        'mean_goes_frp_sum_at_candidate_pos': float(dfx.loc[dfx['label_ignited_next_day'] == 1, 'goes_frp_sum_at_candidate'].mean()),
        'mean_goes_frp_sum_at_candidate_neg': float(dfx.loc[dfx['label_ignited_next_day'] == 0, 'goes_frp_sum_at_candidate'].mean()),
    })
if compare_rows:
    display(pd.DataFrame(compare_rows))
else:
    print('No h2/h4/h6 comparison files found for this fire/date.')